<a href="https://colab.research.google.com/github/sp-zen/DO-1st-RUBYframwrk/blob/main/mlCreateTestCasesWhuggingface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --------------------------------------------------------------
# 1️⃣  Enable GPU (make sure Runtime → Change runtime type → GPU)
import torch, warnings
warnings.filterwarnings('ignore')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device detected: {device}')

In [ ]:
# 2️⃣  Install libraries
!pip install -qU transformers accelerate sentencepiece


In [ ]:
# 3️⃣  Imports
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline


In [ ]:
# 4️⃣  Choose a model (fits on free‑tier GPU)
model_name = "bigscience/bloom-560m"   # change if you have a bigger GPU
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

generator = pipeline(
    "text2text-generation",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
    max_length=512,
    temperature=0.7,
    do_sample=True,
    top_p=0.95,
    repetition_penalty=1.1
)


In [ ]:
# 5️⃣  Prompt builder
def build_prompt(req):
    return (
        "You are a QA engineer. Given the following requirement, generate a list of test cases. "
        "Use the format: \n"
        "- TestID: <id>\n"
        "  Description: <short description>\n"
        "  Preconditions: <any required state>\n"
        "  ExpectedResult: <what should happen>\n"
        f"Requirement: \"{req}\""
    )

In [ ]:
# 6️⃣  Example requirement
requirement = (
    "When a user clicks the ‘Reset Password’ link on the login page, "
    "the system must send a password‑reset email containing a secure token, "
    "and the token must expire after 30 minutes."
)

prompt = build_prompt(requirement)
print("=== Prompt ===")
print(prompt)

print("\n=== Generated test cases ===")
out = generator(prompt, num_return_sequences=1)[0]["generated_text"]
print(out)